# Float Precision vs. Decimal Module for Financial Transactions

This notebook demonstrates the precision issues of standard Python floats when handling financial transactions and shows how the `decimal` module provides perfect accuracy. Run this notebook cell by cell, from top to bottom.

## The Problem: Imperfect Floats

We'll start by simulating one million financial transactions using standard Python floats. These floats, while efficient, are approximations and can lead to tiny, accumulating errors, especially with common decimal values like 0.1 or 0.01.

In [ ]:
import random
import csv

# Set the number of transactions to simulate
NUM_TRANSACTIONS = 1_000_000

# 1. Generate one million random transaction amounts as standard floats
# Each amount is between 0.01 and 10.00, rounded to two decimal places.
print(f"Generating {NUM_TRANSACTIONS} random float transactions...")
transactions_float = [
    round(random.uniform(0.01, 10.00), 2)
    for _ in range(NUM_TRANSACTIONS)
]
print(f"First 5 float transactions: {transactions_float[:5]}")

In [ ]:
# 3. Calculate the sum of these floats directly
# This is our initial, seemingly correct total.
total_float_direct = sum(transactions_float)
print(f"Total (float, direct sum): {total_float_direct:.2f}")

In [ ]:
# 4. Simulate a data reload: convert each float to a string and then back to a float
# This mimics how data might be stored in a database or file and then re-read.
# Any small inaccuracies in the float representation can be exposed or exacerbated here.
print("Simulating data reload for floats (float -> string -> float)...")
transactions_float_reloaded = [
    float(str(t))
    for t in transactions_float
]

# Calculate the sum again after reprocessing
total_float_reprocessed = sum(transactions_float_reloaded)
print(f"Total (float, reprocessed sum): {total_float_reprocessed:.2f}")

In [ ]:
# 5. Print both sums and their difference, clearly showing any discrepancy.
# This difference highlights the insidious nature of float precision errors.
difference_float = total_float_direct - total_float_reprocessed
print(f"\nDifference between direct and reprocessed float sums: {difference_float:.10f}")
if difference_float != 0:
    print("\nNotice the discrepancy! Even a 'penny' difference over a million transactions can add up.")
else:
    print("No discrepancy observed in this run, which can happen due to randomness.")

## The Solution: The `Decimal` Module

Now, we'll repeat the process using Python's `decimal` module. This module provides arbitrary-precision decimal floating-point arithmetic, which is crucial for financial calculations where exact precision is paramount, even if it means a slight performance or memory trade-off.

In [ ]:
from decimal import Decimal, getcontext

# Configure the decimal context for two-decimal precision for financial calculations.
# We set a higher precision for internal calculations (e.g., 28) and then format to 2.
# Using getcontext().prec = 2 directly can lead to premature rounding during intermediate calculations.
# Instead, we ensure the initial Decimal object is created from the string representation
# of the rounded float to capture the intended two-decimal value accurately.
getcontext().prec = 28 # Sufficient precision for general calculations

# 6. Generate one million random transaction amounts, converting immediately to Decimal
# We convert the `round(random.uniform(...))` result to a string first to ensure precise
# representation when creating the Decimal object.
print(f"\nGenerating {NUM_TRANSACTIONS} random Decimal transactions...")
transactions_decimal = [
    Decimal(str(round(random.uniform(0.01, 10.00), 2)))
    for _ in range(NUM_TRANSACTIONS)
]
print(f"First 5 Decimal transactions: {transactions_decimal[:5]}")

In [ ]:
# Calculate the sum of these Decimal objects directly
# This is our initial sum using the precise Decimal type.
total_decimal_direct = sum(transactions_decimal)
print(f"Total (Decimal, direct sum): {total_decimal_direct:.2f}")

In [ ]:
# Simulate a data reload for Decimals: convert each Decimal to a string and then back to a Decimal
# This tests the stability of Decimal representation through common data handling steps.
print("Simulating data reload for Decimals (Decimal -> string -> Decimal)...")
transactions_decimal_reloaded = [
    Decimal(str(t))
    for t in transactions_decimal
]

# Calculate the sum again after reprocessing
total_decimal_reprocessed = sum(transactions_decimal_reloaded)
print(f"Total (Decimal, reprocessed sum): {total_decimal_reprocessed:.2f}")

In [ ]:
# Print both Decimal sums and their difference.
# We expect this difference to be exactly zero, demonstrating perfect precision.
difference_decimal = total_decimal_direct - total_decimal_reprocessed
print(f"\nDifference between direct and reprocessed Decimal sums: {difference_decimal:.10f}")
if difference_decimal == 0:
    print("\nAs expected, no discrepancy with Decimal objects! Precision maintained.")
else:
    print("Unexpected discrepancy with Decimal objects. Check precision settings.")

## Saving Results and Acknowledging Trade-offs

While `Decimal` objects offer superior precision, they generally consume more memory than standard floats. This demonstration prioritizes precision, but in real-world applications, memory usage is an important consideration.

In [ ]:
# Save the original float transactions to a CSV file
float_output_filename = 'float_transactions.csv'
with open(float_output_filename, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['amount'])
    for amount in transactions_float:
        writer.writerow([amount])
print(f"Saved original float transactions to {float_output_filename}")

# Save the Decimal transactions to a CSV file
decimal_output_filename = 'decimal_transactions.csv'
with open(decimal_output_filename, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['amount'])
    for amount in transactions_decimal:
        # Convert Decimal to string for accurate CSV storage
        writer.writerow([str(amount)])
print(f"Saved Decimal transactions to {decimal_output_filename}")

## Conclusion

You've successfully run a simulation demonstrating the critical difference between standard floating-point numbers and Python's `Decimal` objects for financial calculations. While `Decimal` objects have a larger memory footprint, their perfect precision is invaluable when accuracy is non-negotiable.

Your generated transaction data has been saved to `float_transactions.csv` and `decimal_transactions.csv`.

Next time, we'll explore ways to optimize memory usage while maintaining precision, potentially using libraries like NumPy for more efficient storage.